<a href="https://colab.research.google.com/github/JakeOh/202605_BD57/blob/main/lab_python/da12_shape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DataFrame의 모양(shape) 변경

wide(column이 많은) DF <---> long(row이 많은) DF

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

# stack vs unstack

In [2]:
df = pd.DataFrame(data=np.arange(1, 7).reshape((2, 3)),
                  columns=['a', 'b', 'c'],
                  index=['X', 'Y'])
df

,a,b,c
X,1,2,3
Y,4,5,6


In [3]:
df_stacked = df.stack()  #> 컬럼의 이름들이 행 인덱스로 설정. stack: column --> row
df_stacked

X  a    1
   b    2
   c    3
Y  a    4
   b    5
   c    6
dtype: int64

In [4]:
df_stacked.index.nlevels

2

In [7]:
df_unstack1 = df_stacked.unstack()  #> 행 인덱스가 컬럼 이름으로 변환. row ---> column
df_unstack1

,a,b,c
X,1,2,3
Y,4,5,6


In [9]:
df_unstack2 = df_stacked.unstack(level=0)  # 첫번째 레벨의 행 인덱스가 컬럼 이름으로 변환. row ---> column
df_unstack2

,X,Y
a,1,4
b,2,5
c,3,6


## 컬럼 이름이 계층적(다중) 인덱스인 경우

In [10]:
df = pd.DataFrame(data=np.arange(1, 13).reshape((2, 6)),
                  columns=[['Fri', 'Fri', 'Sat', 'Sat', 'Sun', 'Sun'],
                           ['Lunch', 'Dinner'] * 3])
df

Fri          Sat          Sun       
  Lunch Dinner Lunch Dinner Lunch Dinner
0     1      2     3      4     5      6
1     7      8     9     10    11     12

In [11]:
df.columns  #> MultiIndex - 튜플들의 배열

MultiIndex([('Fri',  'Lunch'),
            ('Fri', 'Dinner'),
            ('Sat',  'Lunch'),
            ('Sat', 'Dinner'),
            ('Sun',  'Lunch'),
            ('Sun', 'Dinner')],
           )

In [13]:
df.stack(future_stack=True)
#> 가장 마지막 레벨의 컬럼 이름들을 행 인덱스로 변환. columns ---> rows
#> level=-1 (기본값)

Fri  Sat  Sun
0 Lunch     1    3    5
  Dinner    2    4    6
1 Lunch     7    9   11
  Dinner    8   10   12

In [16]:
df.stack(level=0, future_stack=True)
#> 첫번째 레벨의 컬럼 이름들이 행 인덱스로 변환. columns ---> rows.

Lunch  Dinner
0 Fri      1       2
  Sat      3       4
  Sun      5       6
1 Fri      7       8
  Sat      9      10
  Sun     11      12

# pivot vs melt

In [18]:
df = pd.DataFrame(data={
    'time': ['Lunch'] * 3 + ['Dinner'] * 3,
    'day': ['Fri', 'Sat', 'Sun'] * 2,
    'tip': np.arange(1, 7),
    'total_bill': np.arange(10, 70, 10),
})
df

,time,day,tip,total_bill
0,Lunch,Fri,1,10
1,Lunch,Sat,2,20
2,Lunch,Sun,3,30
3,Dinner,Fri,4,40
4,Dinner,Sat,5,50
5,Dinner,Sun,6,60


# `pandas.DataFrame.pivot()`

*   카테고리(범주) 타입 컬럼의 값들을 컬럼 이름 또는 행 인덱스로 변환.
*   파라미터:
    *   `columns`: pivoting된 데이터프레임에서 컬럼 이름으로 사용하기 위한 변수 이름(들).
    *   `index`: pivoting된 데이터프레임에서 행 인덱스로 사용하기 위한 변수 이름(들).
    *   `values`: pivoting된 데이터프레임에서 각 셀을 채우기 위한 값들을 가지고 있는 변수 이름(들).


In [20]:
df.pivot(columns='day', index='time', values='tip')

day,Fri,Sat,Sun
time,,,
Dinner,4,5,6
Lunch,1,2,3


In [22]:
df.pivot(columns='time', index='day', values='tip')

time,Dinner,Lunch
day,,
Fri,4,1
Sat,5,2
Sun,6,3


In [26]:
df.pivot(columns='day', index='time', values=['tip', 'total_bill'])

tip         total_bill        
day    Fri Sat Sun        Fri Sat Sun
time                                 
Dinner   4   5   6         40  50  60
Lunch    1   2   3         10  20  30

In [27]:
df.pivot(columns='time', index='day', values=['tip', 'total_bill'])

tip       total_bill      
time Dinner Lunch     Dinner Lunch
day                               
Fri       4     1         40    10
Sat       5     2         50    20
Sun       6     3         60    30